In [ ]:
import random
from datetime import datetime

In [ ]:
class OrderProcessor:
    def __init__(self, tax_rate=0.08):
        self.tax_rate = tax_rate
        self.processed_orders = []
        self.failed_orders = []

    def generate_sample_data(self, count=10):
        categories = ["Electronics", "Books", "Clothing", "Home"]
        raw_orders = []
        for i in range(1, count + 1):
            order = {
                "order_id": f"ORD-{1000 + i}",
                "category": random.choice(categories),
                "price": round(random.uniform(10.0, 200.0), 2),
                "quantity": random.randint(1, 5),
                "discount_code": random.choice(["SAVE10", "VIP20", None, "INVALID"])
            }
            raw_orders.append(order)
        # Injecting an edge-case order to trigger a bug
        raw_orders.append({
            "order_id": "ORD-9999",
            "category": "Electronics",
            "price": 150.0,
            "quantity": 2,
            "discount_code": "VIP20"
        })
        return raw_orders

    def calculate_discount(self, subtotal, code):
        if code == "SAVE10":
            return subtotal * 0.10
        elif code == "VIP20":
            # BUG 2: Logic error (adds 20% instead of subtracting, or wrong rate)
            return subtotal * 0.2
        return 0.0

    def process_single_order(self, order):
        price = order["price"]
        qty = order["quantity"]
        
        # Will fail with TypeError when price is a string ("150.00")
        subtotal = price * qty
        discount = self.calculate_discount(subtotal, order["discount_code"])
        
        taxable_amount = subtotal - discount
        tax = taxable_amount * self.tax_rate
        final_total = round(taxable_amount + tax, 2)

        record = {
            "id": order["order_id"],
            "category": order["category"],
            "final_total": final_total,
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }
        self.processed_orders.append(record)
        return record

    def run_batch(self, orders):
        print(f"Starting batch processing for {len(orders)} orders...")
        for order in orders:
            self.process_single_order(order)
        print("Batch processing complete.")

    def get_category_averages(self, target_category):
        category_totals = [
            o["final_total"] for o in self.processed_orders 
            if o["category"] == target_category
        ]
        # BUG 3: ZeroDivisionError if target_category has 0 orders
        if not category_totals:
            return 0.0
        average = sum(category_totals) / len(category_totals)
        return round(average, 2)

In [ ]:
def main():
    random.seed(42)
    processor = OrderProcessor(tax_rate=0.08)
    
    # Step 1: Generate data
    orders = processor.generate_sample_data(count=8)
    
    # Step 2: Process orders (Will crash on ORD-9999 due to Bug 1)
    processor.run_batch(orders)
    
    # Step 3: Analyze results (Will crash if "Groceries" is queried due to Bug 3)
    electronics_avg = processor.get_category_averages("Electronics")
    print(f"Average Electronics Order: ${electronics_avg}")
    
    grocery_avg = processor.get_category_averages("Groceries")
    print(f"Average Grocery Order: ${grocery_avg}")



In [ ]:
main()